# APK 6725 - AI for Sport & Movement Sciences
## Assignment 10 (final assignment): From Images and Videos to Movement Measures

This assignment practices the ideas from lectures 06.01 to 08.02.

- **Part A (2.5 points) - images and video (06–07):**
  - A1: classify postures in images with a pretrained CNN
  - A2: track a barbell in a video and compute the bar velocity
- **Part B (2.5 points) - pose estimation (08):**
  - B1: knee angle during walking, from a gait video
  - B2: can a normal video camera replace the gait laboratory? (cerebral palsy data)

As in Assignment 9, **all the code is given**. Your work is to run the cells, look at the results, and **explain** them by completing the sentences in the answer cells. Use **your** numbers: they can be a little different from those of other students.

**Before you start:** in Colab, use a GPU if you can (Runtime > Change runtime type > T4 GPU). With a GPU the notebook takes about 3 minutes; with a CPU about 8 minutes.

Total: 5 points.

## Words you need

| Word | Meaning |
| --- | --- |
| **pretrained model** | a model trained by other people on a very large dataset (for example ImageNet), that we reuse |
| **embedding** | the vector of numbers that a CNN computes for an image, before its last layer (512 numbers for ResNet18) |
| **object detection** | find each object in an image, with a box and a class |
| **keypoint** | a body landmark (for example the knee) found by a pose estimation model, with an (x, y) position in pixels and a confidence |
| **heel strike** | the moment when the foot touches the ground; one **stride** = from one heel strike to the next heel strike of the same foot |
| **GDI** | Gait Deviation Index: 100 = typical gait, lower = more deviation from typical gait (from motion capture) |
| **r** | correlation between the predicted and the true values (1 = perfect) |

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/Assignments'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
DATA = REPO + '/lectures/data'        # the folder that holds the data files
print('Setup complete. You can now run the rest of the notebook.')

## How to submit

- Run all the cells in order (or **Runtime > Run all**).
- Write your answers in the answer cells. Short answers are good: 1 or 2 sentences for each item.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_10.ipynb`, and upload it to Canvas.

In [ ]:
!pip install -q ultralytics
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image
from scipy.signal import medfilt, savgol_filter, find_peaks
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from ultralytics import YOLO

MODELS = REPO + '/models'
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)
print('Ready. Device:', device)

---
# Part A - Images and video (2.5 points)

## Question A1 (1 point) - Posture images: raw pixels or a pretrained CNN?

The data is the posture dataset of lecture 06.03: images of people **lying, sitting or standing**. We use the `train` folder to train and the `valid` folder as the **test** set.

> **What this experiment does:** It trains two logistic regressions (module 04) on the same images, with two different inputs:
1. **raw pixels**: each image is made smaller (32 × 32 pixels, color), so each image is 32 × 32 × 3 = 3,072 numbers;
2. **ResNet18 embeddings**: each image goes through a ResNet18 pretrained on ImageNet, without its last layer, so each image is 512 numbers.

The CNN is **not** trained here: we only use what it learned on ImageNet. This takes about 1 to 2 minutes.

In [ ]:
from torchvision import datasets, transforms, models
train_dir, test_dir = f'{DATA}/posture-classification/train', f'{DATA}/posture-classification/valid'
classes = datasets.ImageFolder(train_dir).classes

# 1) raw pixels: 32 x 32 color images, flattened
def load_pixels(folder):
    ds = datasets.ImageFolder(folder)
    X = np.array([np.asarray(Image.open(p).convert('RGB').resize((32, 32))).ravel() / 255 for p, _ in ds.samples])
    y = np.array([label for _, label in ds.samples])
    return X, y
Xp_train, y_train = load_pixels(train_dir)
Xp_test, y_test = load_pixels(test_dir)

# 2) ResNet18 embeddings (ImageNet normalization, as in 06.03)
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()                     # remove the last layer -> 512-number embedding
resnet = resnet.to(device).eval()
@torch.no_grad()
def embed(folder):
    loader = torch.utils.data.DataLoader(datasets.ImageFolder(folder, transform=tf), batch_size=32)
    return np.concatenate([resnet(x.to(device)).cpu().numpy() for x, _ in loader])
Xe_train, Xe_test = embed(train_dir), embed(test_dir)
print('Raw pixels:', Xp_train.shape, '| Embeddings:', Xe_train.shape)

baseline = np.bincount(y_test).max() / len(y_test)
results_A1 = {}
for name, (Xtr, Xte) in {'raw pixels': (Xp_train, Xp_test), 'ResNet18 embeddings': (Xe_train, Xe_test)}.items():
    model = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(C=0.1, max_iter=3000))])
    model.fit(Xtr, y_train)
    results_A1[name] = model.predict(Xte)
print(f'Baseline (always "{classes[np.bincount(y_test).argmax()]}"): {baseline:.3f}')
for name, pred in results_A1.items():
    print(f'Logistic regression on {name:20s}: test accuracy = {accuracy_score(y_test, pred):.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (name, pred) in zip(ax, results_A1.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=classes, cmap='Blues', ax=a, colorbar=False)
    a.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# test images that the embedding model got wrong
paths = [p for p, _ in datasets.ImageFolder(test_dir).samples]
wrong = np.flatnonzero(results_A1['ResNet18 embeddings'] != y_test)
plt.figure(figsize=(14, 3))
for k, i in enumerate(wrong[:6]):
    plt.subplot(1, 6, k + 1); plt.imshow(Image.open(paths[i]).convert('RGB'))
    plt.title(f"true: {classes[y_test[i]]}\npred: {classes[results_A1['ResNet18 embeddings'][i]]}", fontsize=9); plt.axis('off')
plt.show()

> **What to look for:** the three accuracies (baseline, raw pixels, embeddings), the confusion matrices, and the wrong images: is the posture easy to see in them?

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The baseline accuracy is ___. With raw pixels the test accuracy is ___, and with the ResNet18 embeddings it is ___.
2. The embeddings work much better than the raw pixels, because ...
3. The postures that the embedding model confuses most are ___ and ___. Looking at the wrong images, a reason is ...

## Question A2 (1.5 points) - Bar velocity from a weightlifting video

We use the barbell detector of lecture 07.02b on the video `lift_two.mp4`. In this video, the lifter does a **clean and jerk**: first the bar goes from the floor to the shoulders (the clean), then from the shoulders to above the head (the jerk).

> **What this experiment does:** It detects the barbell in each frame, then cleans the detections with two physical rules: (1) the plate has a fixed size (diameter 0.45 m), and (2) the bar cannot jump between frames. It converts pixels to meters with the plate size, smooths the height, and computes the vertical velocity. It also repeats the analysis **without** rule 1, to see why the cleaning matters. This takes about 1 minute.

In [ ]:
barbellmodel = YOLO(f'{MODELS}/barbelldetector.pt')

def detect_barbell(video_path, x_min=600, x_max=1500):
    video = cv2.VideoCapture(video_path); fps = video.get(cv2.CAP_PROP_FPS); detections = []
    while True:
        valid, frame = video.read()
        if not valid: break
        r = barbellmodel(frame[:, x_min:x_max], verbose=False)[0]            # YOLO takes the BGR frame directly
        detections.append(np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()] if len(r.boxes) else np.zeros((0, 5)))
    video.release()
    return detections, fps

def clean_track(detections, use_size_rule=True):
    heights = np.concatenate([d[:, 3] - d[:, 1] for d in detections])
    plate_px = np.median(heights[heights > 150])
    cy, prev = [], None
    for d in detections:
        ok = d[np.abs((d[:, 3] - d[:, 1]) - plate_px) < 0.25 * plate_px] if use_size_rule else d   # rule 1
        if len(ok) == 0: cy.append(np.nan); continue
        c = (ok[:, 1] + ok[:, 3]) / 2
        k = np.argmax(ok[:, 4]) if prev is None else np.argmin(np.abs(c - prev))                   # rule 2
        prev = c[k]; cy.append(prev)
    cy = np.array(cy); ok = np.isfinite(cy)
    cy[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), cy[ok])
    return cy, plate_px

def height_velocity(cy, plate_px, fps):
    height = -(cy - cy[0]) * 0.45 / plate_px                                  # meters, up = positive
    height = savgol_filter(medfilt(height, 5), 9, 2)
    return height, savgol_filter(height, 9, 2, deriv=1, delta=1 / fps)

detections, fps = detect_barbell(f'{DATA}/lift_two.mp4')
cy, plate_px = clean_track(detections)
height, velocity = height_velocity(cy, plate_px, fps)
cy_no_rule, _ = clean_track(detections, use_size_rule=False)
height_nr, velocity_nr = height_velocity(cy_no_rule, plate_px, fps)
t = np.arange(len(cy)) / fps

lifts, _ = find_peaks(velocity, height=0.5, distance=int(fps))            # upward movements faster than 0.5 m/s
print(f'1 pixel = {1000 * 0.45 / plate_px:.1f} mm | {len(t)} frames at {fps} frames/s')
for k, i in enumerate(lifts):
    print(f'Upward movement {k + 1}: peak velocity = {velocity[i]:.2f} m/s at {t[i]:.2f} s')
print(f'Maximum bar height: {height.max():.2f} m')

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, height_nr, color='gray', alpha=0.6, label='without the size rule')
ax[0].plot(t, height, label='with both rules'); ax[0].set_ylabel('Bar height (m)'); ax[0].legend()
ax[1].plot(t, velocity, color='tab:orange'); ax[1].plot(t[lifts], velocity[lifts], 'x', color='red', markersize=10)
ax[1].axhline(0, color='gray', linewidth=0.8); ax[1].set_ylabel('Vertical velocity (m/s)'); ax[1].set_xlabel('Time (s)')
plt.show()

> **What to look for:** the red crosses (fast upward movements) and their velocities, and the gray line (no size rule): what happens at about 1 s?

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The program found ___ fast upward movements. The peak upward velocity of the clean (the first one) is ___ m/s and of the jerk (the last one) is ___ m/s. The movement in the middle is slower (___ m/s): in the video, it is when the lifter ...
2. Without the size rule, the bar height (gray line) ... This happens because ...
3. The conversion from pixels to meters is correct only if ... (Hint: in which plane is the plate, and what about the camera?)
4. To check if these velocities are correct, I would compare them with ...

---
# Part B - Pose estimation (2.5 points)

## Question B1 (1.5 points) - Knee angle during walking

The video `GaitVideo.mp4` shows a person who walks from the right to the left of the image (a side view, the sagittal plane).

> **What this experiment does:** It runs the YOLO pose model (lecture 08.01) on each frame and keeps the person with the largest box. Then:
1. **Heel strikes:** the foot touches the ground when it is in its most **forward** position relative to the hips. So we look for the peaks of "ankle position minus hip position" in the walking direction.
2. **Knee flexion:** the 2D knee angle (hip–knee–ankle), changed so that 0 degrees = straight leg: flexion = 180 − angle.
3. One full **stride** of the right leg (from one right heel strike to the next) is normalized to 0–100% of the gait cycle, and compared with typical values from the literature.

This takes about 30 s.

In [ ]:
pose_model = YOLO(f'{MODELS}/yolo26s-pose.pt')
video_path = f'{DATA}/GaitVideo.mp4'
cap = cv2.VideoCapture(video_path); fps_g = cap.get(cv2.CAP_PROP_FPS); cap.release()

pose = []
for r in pose_model.predict(video_path, stream=True, verbose=False):
    kp = np.full((17, 2), np.nan)
    if r.boxes is not None and len(r.boxes):
        b = r.boxes.xyxy.cpu().numpy(); k = int(np.argmax((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])))
        conf = r.keypoints.conf.cpu().numpy()[k]
        kp = np.where(conf[:, None] >= 0.3, r.keypoints.xy.cpu().numpy()[k], np.nan)
    pose.append(kp)
pose = np.array(pose); tg = np.arange(len(pose)) / fps_g

def fill_smooth(x):
    x = x.copy(); ok = np.isfinite(x)
    x[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), x[ok])
    return savgol_filter(medfilt(x, 5), 9, 2)
P = {j: np.c_[fill_smooth(pose[:, j, 0]), fill_smooth(pose[:, j, 1])] for j in range(11, 17)}   # hips, knees, ankles

hip_x = (P[11][:, 0] + P[12][:, 0]) / 2
direction = np.sign(hip_x[-1] - hip_x[0])               # -1: walking to the left
def heel_strikes(ankle):
    forward = (P[ankle][:, 0] - hip_x) * direction       # how far the ankle is in front of the hips
    return find_peaks(forward, distance=int(0.6 * fps_g), prominence=20)[0], forward

def knee_flexion(hip, knee, ankle):
    v1, v2 = P[hip] - P[knee], P[ankle] - P[knee]
    ang = np.degrees(np.arccos(np.clip((v1 * v2).sum(1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)), -1, 1)))
    return 180 - ang

hs_R, fwd_R = heel_strikes(16); hs_L, fwd_L = heel_strikes(15)
flex_R, flex_L = knee_flexion(12, 14, 16), knee_flexion(11, 13, 15)
print(f'{len(pose)} frames at {fps_g:.1f} frames/s | missing right ankle in {np.isnan(pose[:, 16, 0]).mean():.0%} of the frames')
print('Right heel strikes (s):', np.round(tg[hs_R], 2), '| left heel strikes (s):', np.round(tg[hs_L], 2))

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(tg, fwd_R, color='tab:red', label='right ankle'); ax[0].plot(tg, fwd_L, color='tab:blue', label='left ankle')
ax[0].plot(tg[hs_R], fwd_R[hs_R], 'v', color='tab:red', markersize=10); ax[0].plot(tg[hs_L], fwd_L[hs_L], 'v', color='tab:blue', markersize=10)
ax[0].set_ylabel('Ankle in front of hips (px)'); ax[0].legend(); ax[0].set_title('Heel strikes (triangles)')
ax[1].plot(tg, flex_R, color='tab:red'); ax[1].plot(tg, flex_L, color='tab:blue')
ax[1].set_ylabel('Knee flexion (deg)'); ax[1].set_xlabel('Time (s)')
plt.show()

In [ ]:
# one full stride of the right leg, normalized to 0-100% of the gait cycle
start, end = hs_R[0], hs_R[1]
cycle = np.linspace(0, 100, 101)
knee_cycle = np.interp(cycle, np.linspace(0, 100, end - start + 1), flex_R[start:end + 1])
stride_time = (end - start) / fps_g
print(f'Right stride: {tg[start]:.2f} s to {tg[end]:.2f} s -> stride time = {stride_time:.2f} s')
print(f'Knee flexion at heel strike: {knee_cycle[0]:.0f} deg | maximum: {knee_cycle.max():.0f} deg at {cycle[knee_cycle.argmax()]:.0f}% of the cycle')

# typical sagittal knee flexion of healthy adults (approximate values from gait textbooks)
typical_pct = [0, 15, 40, 60, 73, 100]
typical_deg = [5, 18, 5, 40, 60, 5]
plt.figure(figsize=(8, 4))
plt.plot(cycle, knee_cycle, linewidth=2, label='video (2D pose estimation)')
plt.plot(typical_pct, typical_deg, 'o--', color='gray', label='typical values (approximate)')
plt.xlabel('Gait cycle (%)'); plt.ylabel('Knee flexion (deg)'); plt.legend(); plt.title('Right knee, one stride')
plt.show()

> **What to look for:** the heel strike times, the stride time, and the two curves in the last plot: the peak in swing (about 60–75% of the cycle), and the small flexion just after heel strike (about 15%, the "loading response").

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The right stride time is ___ s. A typical stride time for adults is about 1.0 to 1.2 s, so this value (is / is not) ___ realistic. A possible reason is ...
2. The maximum knee flexion is ___ degrees at ___% of the gait cycle. Compared with the typical values, the peak in swing is (similar / different) ___.
3. Just after heel strike, the typical knee flexes about 15–20 degrees (the loading response, at about 15% of the cycle). In the video curve, this flexion is (visible / not visible) ___, with ___ degrees at about ___% of the cycle.
4. Give **two** reasons why a 2D knee angle from one video is not the same as the knee angle from 3D motion capture.

## Question B2 (1 point) - Can a video camera replace the gait laboratory?

We use the cerebral palsy data of lecture 08.02 (parts 2 and 3): features computed from the OpenPose keypoints of 3,244 gait videos (1,373 patients), with the GDI measured in the motion-capture laboratory.

> **What this experiment does:** It predicts the GDI with a random forest, using three sets of inputs: (1) the demographics (age, height, mass, BMI), (2) the walking speed measured in the laboratory, (3) the video features. The cross-validation is grouped **by patient** (`GroupKFold`), because many patients have more than one video. It reports r (correlation between predicted and true GDI) and the MAE (mean absolute error, in GDI points). This takes about 1 minute.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

cp = pd.read_csv(f'{DATA}/cp_video_gait/cp_video_features.csv').dropna().reset_index(drop=True)
demo_cols = ['age', 'height', 'mass', 'bmi']
video_cols = [c for c in cp.columns if c not in ['video_id', 'patient_id', 'exam_id', 'diagnosis', 'gmfcs', 'GDI', 'speed', 'cadence'] + demo_cols]
y_gdi = cp['GDI']
rows, preds_B2 = [], {}
for name, cols in {'Demographics': demo_cols, 'Lab speed only': ['speed'], 'Video features': video_cols,
                   'Video features + demographics': video_cols + demo_cols}.items():
    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0)
    p = cross_val_predict(rf, cp[cols], y_gdi, cv=GroupKFold(5), groups=cp['patient_id'])
    preds_B2[name] = p
    rows.append({'inputs': name, 'r': np.corrcoef(p, y_gdi)[0, 1], 'MAE (GDI points)': mean_absolute_error(y_gdi, p)})
table_B2 = pd.DataFrame(rows).set_index('inputs')
print(f'{len(cp)} videos, {cp.patient_id.nunique()} patients | baseline MAE (always the mean GDI): {mean_absolute_error(y_gdi, np.full(len(y_gdi), y_gdi.mean())):.2f}')
table_B2.round(3)

> **What to look for:** the r of the demographics, the lab speed and the video features, and how much the MAE improves compared with the baseline.

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The video features predict GDI with r = ___, the demographics with r = ___, and the lab speed with r = ___.
2. This shows that the video features contain information about the gait that (is / is not) ___ explained by age and body size, and that is more than the walking speed, because ...
3. In lecture 08.02 part 1, the sit-to-stand measures from 89 videos added little to the demographics. Give one reason why the result is so different here.
4. Before using this video model in another clinic, I would check ... (Hint: think about the camera, the patients, and the data used for training.)

---
## Summary of your results

Run this cell to see your main results together.

In [ ]:
summary = pd.DataFrame([
    {'question': 'A1', 'result': 'test accuracy, raw pixels', 'value': accuracy_score(y_test, results_A1['raw pixels'])},
    {'question': 'A1', 'result': 'test accuracy, ResNet18 embeddings', 'value': accuracy_score(y_test, results_A1['ResNet18 embeddings'])},
    *[{'question': 'A2', 'result': f'peak velocity of upward movement {k + 1} (m/s)', 'value': velocity[i]} for k, i in enumerate(lifts)],
    {'question': 'B1', 'result': 'right stride time (s)', 'value': stride_time},
    {'question': 'B1', 'result': 'maximum knee flexion in swing (deg)', 'value': knee_cycle.max()},
    *[{'question': 'B2', 'result': f'GDI r, {k}', 'value': v} for k, v in table_B2['r'].items()],
])
summary.round(3)

---
## Extra (no points)

- In A2, change `use_size_rule=False` and also remove rule 2 (always take the most confident box). How does the signal look?
- In B1, compute the same knee curve for the **left** leg (`hs_L`, `flex_L`). Is it similar to the right leg? Is the left leg closer to or farther from the camera?
- Record a short video of yourself walking from the side (full body visible, 30 frames/s), and run the B1 code on it.